# Food Access Data Preparation — 2019 and 2025

Run this notebook from top to bottom in a fresh Python kernel. It reads three original USDA CSVs, keeps intermediate tables in memory, and writes **only `cleaned_2019.csv` and `cleaned_2025.csv`** to a new output directory. No prior processed files, report folders, or other notebooks are required.

The delivered schema contains 103 columns: the original 97 columns plus six official-FD fields. Both `fd_label` and `FD_2` are exported with independent status and provenance; neither replaces the other. No statistical imputation, train/test splitting, feature selection, scaling, or model training is performed.

Companion documents: [column dictionary](../docs/column_dictionary.csv) for column-level lookup and [processed-data guide](../docs/processed_data_guide.md) for interpretation, limitations, and modeling handoff. Neither is an execution dependency.


## 1. Inputs and Output Location

Dependencies: Python 3.10+, pandas, NumPy, and IPython (normally available in a Jupyter kernel). Set `RAW_DATA_DIR` explicitly if the raw-data folder is not above the notebook. The required folder structure is shown in `SOURCE_FILES` below. Original inputs are read as text to preserve identifiers and source missing-value tokens.

The default output directory is timestamped and created only at export. Existing directories are rejected rather than overwritten. The input files, Chinese notebook, and personal reference are never written.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import hashlib, re
import numpy as np
import pandas as pd
from IPython.display import display

RAW_DATA_DIR = None  # Or Path('/your/path/to/raw_data')
if RAW_DATA_DIR is None:
    RAW_DATA_DIR = next((p/'raw_data' for p in [Path.cwd(), *Path.cwd().parents]
                         if (p/'raw_data').is_dir()), None)
if RAW_DATA_DIR is None:
    raise FileNotFoundError('Set RAW_DATA_DIR to the folder containing the two source datasets.')
RAW_DATA_DIR = Path(RAW_DATA_DIR)
OUTPUT_DIR = Path.cwd()/('cleaned_run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
if OUTPUT_DIR.exists():
    raise FileExistsError('Choose a new OUTPUT_DIR; existing folders are not overwritten.')
SOURCE_FILES = {
    'A19': ('2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv', 'utf-8-sig', 'CensusTract'),
    'G25': ('2025-sram-fara-data/SRAM General Tract Characteristics Data.csv', 'cp1252', 'CensusTract20'),
    'S25': ('2025-sram-fara-data/SRAM Straight Line Distance Data.csv', 'cp1252', 'CensusTract20'),
}
def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
for relative, _, _ in SOURCE_FILES.values():
    if not (RAW_DATA_DIR/relative).is_file():
        raise FileNotFoundError(RAW_DATA_DIR/relative)
input_hashes = {relative: file_hash(RAW_DATA_DIR/relative) for relative, _, _ in SOURCE_FILES.values()}
print('Raw inputs:', RAW_DATA_DIR)
print('Planned output:', OUTPUT_DIR)


Raw inputs: /private/var/folders/4r/zlfp_lm51j11_zp5q_dl55tr0000gn/T/fd2_delivery_run_48my88uu/raw_data
Planned output: /private/var/folders/4r/zlfp_lm51j11_zp5q_dl55tr0000gn/T/fd2_delivery_run_48my88uu/cleaned_run_20261101T042243746517Z


## 2. Read Sources and Validate the 2025 Join

Normalize tract keys to 11-digit strings by restoring a missing leading zero. Reject non-digit, scientific-notation, decimal-suffixed, or incorrectly sized keys. Check uniqueness and coverage of the 50 states plus D.C. General and Straight Line must contain the same 2025 tract keys and matching state/county metadata. Their row order is aligned before constructing unified fields.

These checks establish structural coverage, not equivalence of 2010 and 2020 tract boundaries. The two years are never joined into a tract-level change panel.


In [2]:
def normalize_tract(value):
    value = value.strip()
    if re.fullmatch(r'[0-9]{10,11}', value) is None:
        raise ValueError(f'Invalid tract ID: {value!r}')
    return value.zfill(11)

expected_states = set('01 02 04 05 06 08 09 10 11 12 13 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 44 45 46 47 48 49 50 51 53 54 55 56'.split())
raw, source_summary = {}, []
for name, (relative, encoding, key) in SOURCE_FILES.items():
    table = pd.read_csv(RAW_DATA_DIR/relative, dtype=str, keep_default_na=False, encoding=encoding)
    ids = table[key].map(normalize_tract)
    assert ids.is_unique, f'Duplicate tract keys: {name}'
    assert set(ids.str[:2]) == expected_states, f'Unexpected state coverage: {name}'
    table.index = pd.Index(ids, name='tract_key')
    raw[name] = table
    source_summary.append({'source':name, 'rows':len(table), 'columns':len(table.columns),
                           'states_including_DC':ids.str[:2].nunique(), 'encoding':encoding})
assert set(raw['G25'].index) == set(raw['S25'].index)
raw['S25'] = raw['S25'].reindex(raw['G25'].index)
for field in ['State','County20','County24']:
    assert raw['G25'][field].equals(raw['S25'][field]), field
display(pd.DataFrame(source_summary))
display(pd.DataFrame([{'General_rows':len(raw['G25']), 'Straight_Line_rows':len(raw['S25']),
                       'matched_keys':len(raw['G25']), 'unmatched_keys':0, 'metadata_mismatches':0}]))


,source,rows,columns,states_including_DC,encoding
0,A19,72531,147,51,utf-8-sig
1,G25,84119,27,51,cp1252
2,S25,84119,135,51,cp1252


,General_rows,Straight_Line_rows,matched_keys,unmatched_keys,metadata_mismatches
0,84119,84119,84119,0,0


## 3. Field Definitions and Cross-Year Mapping

The proposal's candidate concepts are represented by 24 fields: eight demographic/housing, seven race/ethnicity, six socioeconomic, and three spatial fields. The seven race/ethnicity counts interpret the proposal's broader wording using source definitions; they are not seven binary flags. The low-access population used to construct the target is a separate 25th concept.

Each concept becomes `original__X` (source text), `clean__X` (nullable number), and `qa__X` (quality status). These are **not three separate model features**. Identifiers and provenance are metadata, not predictors. Common names indicate conceptual correspondence, not identical years, boundaries, or retailer definitions.

| Group | Unified concept | Meaning |
|---|---|---|
| Demographics / housing | `Urban` | Official urban=1 / rural=0 classification; not recalculated from population. |
| Demographics / housing | `population_total` | Total Census population: 2010 / 2020. Zero is retained but cannot be a ratio denominator. |
| Demographics / housing | `occupied_housing_units_total` | Occupied housing units: Census 2010 / 2020. Zero households need not mean zero population. |
| Demographics / housing | `GroupQuartersFlag` | Official flag for group-quarters population share at least 67%; not an FD exclusion. |
| Demographics / housing | `NUMGQTRS` | Group-quarters population count, e.g. dormitories or institutional housing. |
| Demographics / housing | `PCTGQTRS` | Group-quarters share, in percent (0–100), not a 0–1 fraction. |
| Demographics / housing | `TractKids` | Population aged 0–17. |
| Demographics / housing | `TractSeniors` | Population aged 65 and over. |
| Race / ethnicity | `TractWhite` | White population count. |
| Race / ethnicity | `TractBlack` | Black or African American population count. |
| Race / ethnicity | `TractAsian` | Asian population count. |
| Race / ethnicity | `TractNHOPI` | Native Hawaiian / Other Pacific Islander population count. |
| Race / ethnicity | `TractAIAN` | American Indian / Alaska Native population count. |
| Race / ethnicity | `TractOMultir` | Other / multiple-race population count. |
| Race / ethnicity | `TractHispanic` | Hispanic / Latino ethnicity count; overlaps race, so excluded from the six-race sum. |
| Socioeconomic | `LowIncomeTracts` | Official low-income flag; poverty >=20% or the applicable state/metro median-family-income 80% criterion. Retain the official value. |
| Socioeconomic | `PovertyRate` | ACS poverty percentage, using the population for whom poverty status is determined; not necessarily total Census population. |
| Socioeconomic | `MedianFamilyIncome` | Median FAMILY income; source price years 2018 / 2024. 2499 and 250001 are endpoint codes, not exact incomes. |
| Socioeconomic | `TractLOWI` | ACS low-income persons (family income at or below 200% of the applicable poverty threshold). Not the poverty count. |
| Socioeconomic | `TractHUNV` | ACS occupied housing units without a vehicle; a household count, not persons. |
| Socioeconomic | `TractSNAP` | ACS households receiving SNAP; not recipients or dollar benefits. |
| Spatial access | `LATracts_half` | Official urban half-mile low-access flag (at least 500 persons OR 33%). Rural zero does not establish good access. |
| Spatial access | `LATracts1` | Official urban one-mile low-access flag (at least 500 persons OR 33%); overlaps the urban FD rule. |
| Spatial access | `LATractsVehicle_20` | Official combined vehicle/distance flag: at least 100 households without a vehicle beyond half a mile OR at least 500 persons/33% beyond 20 miles. Applies to urban and rural tracts. |
| Target source | `low_access_population_1_20` | Low-access persons beyond urban 1 mile / rural 20 miles. Used for fd_label, not automatically a predictor. |

In [3]:
FIELDS = ['Urban', 'population_total', 'occupied_housing_units_total', 'GroupQuartersFlag', 'NUMGQTRS', 'PCTGQTRS', 'TractKids', 'TractSeniors', 'TractWhite', 'TractBlack', 'TractAsian', 'TractNHOPI', 'TractAIAN', 'TractOMultir', 'TractHispanic', 'LowIncomeTracts', 'PovertyRate', 'MedianFamilyIncome', 'TractLOWI', 'TractHUNV', 'TractSNAP', 'LATracts_half', 'LATracts1', 'LATractsVehicle_20', 'low_access_population_1_20']
MAP_2019 = {'Urban': 'Urban', 'population_total': 'Pop2010', 'occupied_housing_units_total': 'OHU2010', 'GroupQuartersFlag': 'GroupQuartersFlag', 'NUMGQTRS': 'NUMGQTRS', 'PCTGQTRS': 'PCTGQTRS', 'TractKids': 'TractKids', 'TractSeniors': 'TractSeniors', 'TractWhite': 'TractWhite', 'TractBlack': 'TractBlack', 'TractAsian': 'TractAsian', 'TractNHOPI': 'TractNHOPI', 'TractAIAN': 'TractAIAN', 'TractOMultir': 'TractOMultir', 'TractHispanic': 'TractHispanic', 'LowIncomeTracts': 'LowIncomeTracts', 'PovertyRate': 'PovertyRate', 'MedianFamilyIncome': 'MedianFamilyIncome', 'TractLOWI': 'TractLOWI', 'TractHUNV': 'TractHUNV', 'TractSNAP': 'TractSNAP', 'LATracts_half': 'LATracts_half', 'LATracts1': 'LATracts1', 'LATractsVehicle_20': 'LATractsVehicle_20', 'low_access_population_1_20': 'LAPOP1_20'}
MAP_2025 = {'Urban': 'Urban', 'population_total': 'POP2020', 'occupied_housing_units_total': 'OHU2020', 'GroupQuartersFlag': 'GroupQuartersFlag', 'NUMGQTRS': 'NUMGQTRS', 'PCTGQTRS': 'PCTGQTRS', 'TractKids': 'TractKids', 'TractSeniors': 'TractSeniors', 'TractWhite': 'TractWhite', 'TractBlack': 'TractBlack', 'TractAsian': 'TractAsian', 'TractNHOPI': 'TractNHOPI', 'TractAIAN': 'TractAIAN', 'TractOMultir': 'TractOMultir', 'TractHispanic': 'TractHispanic', 'LowIncomeTracts': 'LowIncomeTracts', 'PovertyRate': 'PovertyRate', 'MedianFamilyIncome': 'MedianFamilyIncome', 'TractLOWI': 'TractLOWI', 'TractHUNV': 'TractHUNV', 'TractSNAP': 'TractSNAP', 'LATracts_half': 'SD_SRAM_LATracts_half', 'LATracts1': 'SD_SRAM_LATracts1', 'LATractsVehicle_20': 'SD_SRAM_LATractsVehicle_20', 'low_access_population_1_20': 'SD_SRAM_LAPOP1_20'}
base = {}
mapping_rows = []
for year, names, field_map in [('2019',['A19'],MAP_2019), ('2025',['G25','S25'],MAP_2025)]:
    first = raw[names[0]]
    table = pd.DataFrame({'tract_id':first.index.to_numpy(), 'State':first.State.to_numpy(),
                          'County':first['County' if year=='2019' else 'County20'].to_numpy()})
    for field in FIELDS:
        source_field = field_map[field]
        locations = [name for name in names if source_field in raw[name].columns]
        assert locations, (year,field)
        values = raw[locations[0]][source_field]
        for duplicate in locations[1:]:
            assert values.equals(raw[duplicate][source_field])
        table[field] = values.to_numpy()
        mapping_rows.append({'year':year,'concept':field,'source':locations[0],'source_field':source_field})
    official_name = 'LA1and20' if year=='2019' else 'SD_SRAM_LA1and20'
    table['official_low_access_flag'] = raw['A19' if year=='2019' else 'S25'][official_name].to_numpy()
    table['meta_dataset_version'] = year
    table['meta_population_census_year'] = '2010' if year=='2019' else '2020'
    table['meta_tract_vintage'] = '2010' if year=='2019' else '2020'
    base[year] = table
mapping = pd.DataFrame(mapping_rows)
display(mapping)


,year,concept,source,source_field
0,2019,Urban,A19,Urban
1,2019,population_total,A19,Pop2010
2,2019,occupied_housing_units_total,A19,OHU2010
3,2019,GroupQuartersFlag,A19,GroupQuartersFlag
4,2019,NUMGQTRS,A19,NUMGQTRS
5,2019,PCTGQTRS,A19,PCTGQTRS
6,2019,TractKids,A19,TractKids
7,2019,TractSeniors,A19,TractSeniors
8,2019,TractWhite,A19,TractWhite
9,2019,TractBlack,A19,TractBlack


## 4. FD Label Definitions and Comparison

### Formula-based `fd_label` (exported)

Let L be persons beyond **1 mile in urban tracts or 20 miles in rural tracts**, and P be total Census population. For valid P>0 and 0≤L≤P:

**FD = 1 if L ≥ 500 OR 100×L ≥ 33×P; otherwise FD = 0.**

The 500 threshold refers to low-access persons, not total population. Both boundaries are inclusive. No low-income condition or GQ exclusion is added. First validate P, then handle P=0, then validate L. Invalid/missing inputs remain unknown; P=0 is not applicable. Decimal arithmetic avoids rounding the threshold before comparison.

Reasons: `both_thresholds`, `share_threshold_only`, `count_threshold_only`, or `below_both_thresholds` for known labels. Missing/invalid inputs keep their own reasons. NA is never relabeled as zero.

### Official `FD_2` (exported alongside the formula label)

Read 2019 `LA1and20` and 2025 `SD_SRAM_LA1and20` directly. These official flags use the same low-access concept, but need not reproduce calculations from published rounded/missing counts. They are not the low-income-and-low-access (LILA) measure or three model-risk categories.

2019 has 35,914 formula-unknown records whose official flag is zero. This provides a different label source; it does **not** recover their missing L values. Seven comparable records have formula=1 and official=0; the exact upstream precision/generation cause remains unresolved. In 2025, official missing values correspond to 271 zero-population tracts. Never mix the two label populations when reporting coverage.

USDA supplies multiple distance measures; the proposal chose 1/20 miles. Urban/rural distances reflect different spatial contexts, not a guarantee of affordability, transport, or access within the threshold. Retailer-definition differences are retained as a project limitation; 2025 uses Straight Line, not Driving.

Sources: [USDA Large Retailer Access guide](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/large-retailer-access-map-reference-guide), [2025 SNAP-authorized Retailer Access guide](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/snap-authorized-retailer-access-map-reference-guide), and the proposal's Data Modeling section.


The six appended columns are `FD_2_original` (published source text), `FD_2` (nullable 0/1), `FD_2_source_field`, `FD_2_status`, `FD_2_reason`, and `FD_2_comparison`. `not_comparable` means at least one label is unavailable, not that they agree. `label_status` and `label_reason` still describe the formula only; always use the status belonging to the chosen target.


In [4]:
missing_tokens = {'','NULL','NA','N/A','NAN'}
def parse_number(value):
    if str(value).strip().upper() in missing_tokens:return None,'missing'
    try:n=Decimal(str(value).strip())
    except InvalidOperation:return None,'unparseable'
    if not n.is_finite():return None,'nonfinite'
    if n<0:return n,'negative'
    return n,'valid'

def decide_fd(p_raw,l_raw):
    p,ps=parse_number(p_raw);l,ls=parse_number(l_raw)
    result={'fd_label':'','label_status':'unknown','label_reason':'','fd_population_input_status':ps,'fd_low_access_input_status':ls,'fd_share_condition':'unknown','fd_count_condition':'unknown','label_boundary_review':'False'}
    if ps!='valid':result['label_reason']='population_'+ps;return result
    if p==0:
        result.update(label_status='not_applicable',label_reason='zero_population');return result
    if ls!='valid':result['label_reason']='low_access_'+ls;return result
    if l>p:result['label_reason']='low_access_above_population';return result
    share=100*l>=33*p;count=l>=500
    result.update(fd_label='1' if share or count else '0',label_status='known',label_reason='both_thresholds' if share and count else 'share_threshold_only' if share else 'count_threshold_only' if count else 'below_both_thresholds',fd_share_condition=str(share),fd_count_condition=str(count),label_boundary_review=str(abs(l-500)<=Decimal('.5') or abs(l-Decimal('.33')*p)<=Decimal('.5')))
    return result
# Boundary tests do not read or alter source records.
for p_value,l_value,expected in [('1000','330','1'),('2000','500','1'),('1000','329','0'),
                                ('1000','',''),('0','0',''),('1000','1001','')]:
    assert decide_fd(p_value,l_value)['fd_label'] == expected
fd_summary, fd_reason_rows, fd_differences, fd_cross_tables = [], [], [], []
fd_alternative_tables = {}
for year, table in base.items():
    labels = pd.DataFrame([decide_fd(p,l) for p,l in zip(table.population_total,table.low_access_population_1_20)])
    for col in ['fd_label','label_status','label_reason']:
        table[col] = labels[col]
    raw_flag = table.official_low_access_flag
    absent = raw_flag.str.strip().str.upper().isin(missing_tokens)
    flag = pd.to_numeric(raw_flag.mask(absent),errors='coerce')
    valid = ~absent & flag.isin([0,1])
    zero_pop = pd.to_numeric(table.population_total,errors='coerce').eq(0)
    alternate = table[['tract_id','fd_label','label_status','label_reason']].copy()
    alternate['FD_2_original'] = raw_flag
    alternate['FD_2'] = flag.where(valid).astype('Int64')
    alternate['FD_2_source_field'] = 'LA1and20' if year=='2019' else 'SD_SRAM_LA1and20'
    alternate['FD_2_status'] = np.where(valid,'known',np.where(absent & zero_pop,'not_applicable','unknown'))
    alternate['FD_2_reason'] = np.where(valid,'official_flag_available',np.where(absent & zero_pop,
        'zero_population_official_missing',np.where(absent,'official_flag_missing','official_flag_invalid')))
    comparable = labels.label_status.eq('known') & valid
    different = comparable & pd.to_numeric(labels.fd_label,errors='coerce').ne(flag)
    alternate['FD_2_comparison'] = np.where(comparable,np.where(different,'different','agree'),'not_comparable')
    fd_alternative_tables[year] = alternate
    for (status,reason),count in labels.groupby(['label_status','label_reason']).size().items():
        fd_reason_rows.append({'year':year,'status':status,'reason':reason,'rows':count})
    fd_summary.append({'year':year,'rows':len(table),'formula_0':int(labels.fd_label.eq('0').sum()),
        'formula_1':int(labels.fd_label.eq('1').sum()),'formula_unknown':int(labels.label_status.eq('unknown').sum()),
        'formula_not_applicable':int(labels.label_status.eq('not_applicable').sum()),
        'FD_2_0':int(alternate.FD_2.eq(0).sum()),'FD_2_1':int(alternate.FD_2.eq(1).sum()),
        'FD_2_NA':int(alternate.FD_2.isna().sum()),'both_known':int(comparable.sum()),'differences':int(different.sum())})
    cross = pd.crosstab(labels.fd_label.replace('','NA'),alternate.FD_2.astype('string').fillna('NA'))
    cross.index.name = 'formula_fd_label';cross.columns.name = 'FD_2';fd_cross_tables.append((year,cross))
    diff = table.loc[different,['tract_id','population_total','low_access_population_1_20','fd_label','official_low_access_flag']].copy()
    diff.insert(0,'year',year);fd_differences.append(diff)
display(pd.DataFrame(fd_summary))
display(pd.DataFrame(fd_reason_rows))
for year,cross in fd_cross_tables:
    print(year);display(cross)
display(pd.concat(fd_differences,ignore_index=True))


,year,rows,formula_0,formula_1,formula_unknown,formula_not_applicable,FD_2_0,FD_2_1,FD_2_NA,both_known,differences
0,2019,72531,11901,24716,35914,0,47822,24709,0,36617,7
1,2025,84119,70493,13355,0,271,70493,13355,271,83848,0


,year,status,reason,rows
0,2019,known,below_both_thresholds,11901
1,2019,known,both_thresholds,17204
2,2019,known,count_threshold_only,7341
3,2019,known,share_threshold_only,171
4,2019,unknown,low_access_missing,35914
5,2025,known,below_both_thresholds,70493
6,2025,known,both_thresholds,7136
7,2025,known,count_threshold_only,6128
8,2025,known,share_threshold_only,91
9,2025,not_applicable,zero_population,271


2019


FD_2,0,1
formula_fd_label,,
0,11901,0
1,7,24709
NA,35914,0


2025


FD_2,0,1,NA
formula_fd_label,,,
0,70493,0,0
1,0,13355,0
NA,0,0,271


,year,tract_id,population_total,low_access_population_1_20,fd_label,official_low_access_flag
0,2019,12099001802,3920,500,1,0
1,2019,13115001100,2782,500,1,0
2,2019,17043840703,3898,500,1,0
3,2019,17201000105,4442,500,1,0
4,2019,27137001800,1839,500,1,0
5,2019,29510105500,2922,500,1,0
6,2019,39049980000,2,1,1,0


## 5. Numeric Cleaning and Field-Level Quality Results

Preserve source text; convert source missing tokens to numeric NA. Invalid numeric text, nonfinite values, and negative values become NA with a reason. Binary flags must be 0/1; percentage fields must be 0–100. Income zero is invalid. Low-access persons above the same-period population are invalid. Valid zeros are otherwise retained.

No general median filling, clipping to a denominator, or whole-row deletion is used. In particular, ACS counts may exceed an older Census reference total because periods differ; this is recorded, not automatically corrected. Endpoint income codes stay in `clean__MedianFamilyIncome` until separate derived-income columns are created.


In [5]:
FLAGS = {'Urban','GroupQuartersFlag','LowIncomeTracts','LATracts_half','LATracts1','LATractsVehicle_20'}
PERCENTAGES = {'PCTGQTRS','PovertyRate'}
METADATA = ['tract_id','State','County','meta_dataset_version','meta_population_census_year','meta_tract_vintage','fd_label','label_status','label_reason']
cleaned, quality_rows = {}, []
for year, table in base.items():
    layer = table[METADATA].copy()
    population = pd.to_numeric(table.population_total,errors='coerce')
    for field in FIELDS:
        source = table[field]
        absent = source.str.strip().str.upper().isin(missing_tokens)
        value = pd.to_numeric(source.str.strip().mask(absent),errors='coerce').astype('Float64')
        qa = pd.Series('valid',index=table.index,dtype='string')
        qa.loc[absent] = 'source_missing'
        qa.loc[~absent & value.isna()] = 'parse_failure'
        finite = pd.Series(np.isfinite(value.to_numpy(dtype=float,na_value=np.nan)),index=table.index)
        qa.loc[value.notna() & ~finite] = 'nonfinite'
        qa.loc[finite & value.lt(0).fillna(False)] = 'negative'
        if field=='MedianFamilyIncome':
            qa.loc[qa.eq('valid') & value.eq(0).fillna(False)] = 'nonpositive_income'
        if field in FLAGS:
            qa.loc[qa.eq('valid') & ~value.isin([0,1])] = 'invalid_binary'
        if field in PERCENTAGES:
            qa.loc[qa.eq('valid') & ~value.between(0,100).fillna(False)] = 'invalid_percentage'
        if field=='low_access_population_1_20':
            qa.loc[qa.eq('valid') & population.notna() & value.gt(population).fillna(False)] = 'above_same_period_population'
        layer['original__'+field] = source
        layer['clean__'+field] = value.mask(~qa.eq('valid'),pd.NA)
        layer['qa__'+field] = qa
        quality_rows.append({'year':year,'field':field,'source_missing':int(absent.sum()),
            'invalid_to_NA':int((~qa.isin(['valid','source_missing'])).sum()),
            'clean_NA':int(layer['clean__'+field].isna().sum()),
            'missing_with_known_fd_label':int((layer['clean__'+field].isna() & table.label_status.eq('known')).sum()),
            'missing_with_known_FD_2':int((layer['clean__'+field].isna() & fd_alternative_tables[year].FD_2_status.eq('known')).sum()),'valid_zero':int(layer['clean__'+field].eq(0).sum())})
    assert len(layer)==len(table)
    for field in FIELDS:
        assert layer['original__'+field].equals(table[field])
    cleaned[year] = layer
quality = pd.DataFrame(quality_rows)
display(quality)


,year,field,source_missing,invalid_to_NA,clean_NA,missing_with_known_fd_label,missing_with_known_FD_2,valid_zero
0,2019,Urban,0,0,0,0,0,17362
1,2019,population_total,0,0,0,0,0,0
2,2019,occupied_housing_units_total,0,0,0,0,0,106
3,2019,GroupQuartersFlag,0,0,0,0,0,72015
4,2019,NUMGQTRS,25,0,25,10,25,27235
5,2019,PCTGQTRS,25,0,25,10,25,27235
6,2019,TractKids,4,0,4,0,4,144
7,2019,TractSeniors,4,0,4,0,4,187
8,2019,TractWhite,4,0,4,0,4,29
9,2019,TractBlack,4,0,4,0,4,423


### 5.1 Relationships That Must Not Be “Fixed” Automatically

GQ count/share missingness cannot be recovered from a 0/1 GQ flag. A zero-household tract can still contain group-quarters residents. Zero-population denominators remain unusable. Six race counts are checked together; Hispanic is excluded from their sum. Counts and percentages are retained rather than automatically turned into new ratios.

Demographic counts use Census 2010/2020; economic counts use ACS 2014–2018/2020–2024. An ACS-to-Census ratio is not automatically a same-period participation rate. Official flags are retained even when a diagnostic reconstruction from rounded published counts differs. This is a source limitation, not a reason to overwrite the flag.

For the vehicle flag, the detailed lookup uses at least 100 households while a narrative source says more than 100. Published boundary cases cannot resolve the exact upstream rule; retain the official flag. See the companion guide for the recorded discrepancies.


In [6]:
relationship_rows = []
race6 = ['TractWhite','TractBlack','TractAsian','TractNHOPI','TractAIAN','TractOMultir']
for year, layer in cleaned.items():
    v = pd.DataFrame({f:layer['clean__'+f] for f in FIELDS})
    complete = v[race6].notna().all(axis=1) & v.population_total.notna()
    metrics = {
        'zero_population':v.population_total.eq(0),
        'zero_occupied_housing_units':v.occupied_housing_units_total.eq(0),
        'positive_population_zero_housing_units':v.population_total.gt(0)&v.occupied_housing_units_total.eq(0),
        'six_race_sum_differs_from_population':complete & v[race6].sum(axis=1).ne(v.population_total),
        'children_plus_seniors_above_population':v.TractKids.add(v.TractSeniors).gt(v.population_total),
        'TractLOWI_above_Census_reference':v.TractLOWI.gt(v.population_total),
        'TractHUNV_above_Census_reference':v.TractHUNV.gt(v.occupied_housing_units_total),
        'TractSNAP_above_Census_reference':v.TractSNAP.gt(v.occupied_housing_units_total),
    }
    for name,mask in metrics.items():
        relationship_rows.append({'year':year,'check':name,'rows':int(mask.fillna(False).sum())})
display(pd.DataFrame(relationship_rows))


,year,check,rows
0,2019,zero_population,0
1,2019,zero_occupied_housing_units,106
2,2019,positive_population_zero_housing_units,106
3,2019,six_race_sum_differs_from_population,0
4,2019,children_plus_seniors_above_population,0
5,2019,TractLOWI_above_Census_reference,131
6,2019,TractHUNV_above_Census_reference,52
7,2019,TractSNAP_above_Census_reference,22
8,2025,zero_population,271
9,2025,zero_occupied_housing_units,420


## 6. Income Price Basis and Endpoint Proxies

The 2019 income estimate uses 2018 dollars; the 2025 estimate uses 2024 dollars. Both are expressed in 2018 dollars using a frozen, previously verified BLS R-CPI-U-RS all-items annual-average ratio. This is a national time adjustment, not a county cost-of-living adjustment.

- Annual indices: 2018 = **369.1**; 2024 = **462.51116666666667**.
- Factor: index2018 / index2024 = **0.7980347861871443** (applied without intentional rounding).
- Source: [BLS workbook](https://www.bls.gov/cpi/research-series/r-cpi-u-rs-allitems.xlsx), verified 2026-10-31. Archived workbook SHA-256: `355ea8a14a6d07d777a17a1ddad526b8a199390a7c1bb8c7b58e2dec1a2544d8`.

The frozen values below avoid requiring a historical report or a live download at runtime. Updating the source vintage requires a deliberate version change.

`2499` and `250001` are source endpoint codes representing income at/below $2,500 or at/above $250,000, not exact dollar estimates. Boundary proxies use 2,500/250,000 in source dollars before conversion; the lower proxy may overestimate and the upper proxy may underestimate actual income.

`income_ordinary_2018_usd` excludes endpoints and missing values. `income_with_endpoint_proxy_2018_usd` includes ordinary amounts and endpoint proxies but leaves genuine missing/invalid values NA. `income_price_factor_applied` is populated only for ordinary income; its NA count is therefore larger than source-income missingness. Endpoint conversion uses the same year's factor.


In [7]:
from decimal import localcontext
with localcontext() as context:
    context.prec = 40
    frozen_factor = Decimal('369.1') / Decimal('462.51116666666667')
PRICE_FACTOR = float(frozen_factor)
income_rows = []
for year, layer in cleaned.items():
    income = layer.clean__MedianFamilyIncome
    qa = layer.qa__MedianFamilyIncome
    missing = qa.eq('source_missing'); invalid = ~qa.isin(['valid','source_missing'])
    lower = qa.eq('valid') & income.eq(2499).fillna(False)
    upper = qa.eq('valid') & income.eq(250001).fillna(False)
    ordinary = qa.eq('valid') & ~lower & ~upper
    status = pd.Series('ordinary',index=layer.index,dtype='string')
    for mask,name in [(missing,'source_missing'),(invalid,'invalid_source'),(lower,'lower_endpoint'),(upper,'upper_endpoint')]:
        status.loc[mask] = name
    assert pd.DataFrame([missing,invalid,lower,upper,ordinary]).sum(axis=0).eq(1).all()
    factor = 1.0 if year=='2019' else PRICE_FACTOR
    layer['income_value_status'] = status
    for name,mask in [('income_source_missing',missing),('income_invalid_source',invalid),
                      ('income_lower_endpoint',lower),('income_upper_endpoint',upper)]:
        layer[name] = mask.astype('Int8')
    layer['income_source_price_year'] = 2018 if year=='2019' else 2024
    layer['income_target_price_year'] = 2018
    layer['income_price_factor_applied'] = pd.Series(factor,index=layer.index,dtype='Float64').where(ordinary,pd.NA)
    layer['income_ordinary_2018_usd'] = (income*factor).where(ordinary,pd.NA).astype('Float64')
    amount = layer.income_ordinary_2018_usd.copy()
    amount.loc[lower] = 2500*factor; amount.loc[upper] = 250000*factor
    layer['income_with_endpoint_proxy_2018_usd'] = amount
    layer['income_endpoint_proxy_applied'] = (lower|upper).astype('Int8')
    layer['income_proxy_method'] = status.map({'ordinary':'ordinary_amount_unchanged',
        'lower_endpoint':'lower_boundary_2500_source_usd','upper_endpoint':'upper_boundary_250000_source_usd',
        'source_missing':'source_missing_unfilled','invalid_source':'invalid_source_unfilled'})
    layer['income_ordinary_reference_candidate'] = ordinary.astype('Int8')
    assert amount.notna().equals((ordinary|lower|upper).astype(bool))
    for name,mask in [('ordinary',ordinary),('lower_endpoint',lower),('upper_endpoint',upper),('source_missing',missing),('invalid_source',invalid)]:
        income_rows.append({'year':year,'status':name,'rows':int(mask.sum()),'price_factor':factor})
display(pd.DataFrame(income_rows))
for year,layer in cleaned.items():
    print(year)
    display(layer.groupby('income_value_status',sort=False).head(1)[['tract_id','original__MedianFamilyIncome',
        'income_value_status','income_ordinary_2018_usd','income_with_endpoint_proxy_2018_usd']])


,year,status,rows,price_factor
0,2019,ordinary,71511,1.000000
1,2019,lower_endpoint,2,1.000000
2,2019,upper_endpoint,270,1.000000
3,2019,source_missing,748,1.000000
4,2019,invalid_source,0,1.000000
5,2025,ordinary,80331,0.798035
6,2025,lower_endpoint,16,0.798035
7,2025,upper_endpoint,1629,0.798035
8,2025,source_missing,2143,0.798035
9,2025,invalid_source,0,0.798035


2019


,tract_id,original__MedianFamilyIncome,income_value_status,income_ordinary_2018_usd,income_with_endpoint_proxy_2018_usd
0,01001020100,81250,ordinary,81250.0,81250.0
105,01015981901,NULL,source_missing,<NA>,<NA>
1707,04013105002,250001,upper_endpoint,<NA>,250000.0
7037,06037980010,2499,lower_endpoint,<NA>,2500.0


2025


,tract_id,original__MedianFamilyIncome,income_value_status,income_ordinary_2018_usd,income_with_endpoint_proxy_2018_usd
0,01001020100,76979,ordinary,61431.919806,61431.919806
74,01007010009,,source_missing,<NA>,<NA>
512,01073004800,250001,upper_endpoint,<NA>,199508.696547
4621,06007001900,2499,lower_endpoint,<NA>,1995.086965


## 7. Final Quality Checks and Handoff Limits

The full table retains every tract. The states of both labels (`known`, `unknown`, `not_applicable`) are reported separately here without exporting separate subsets. A known FD label does not imply complete inputs; the missing-input count uses all 24 candidates only as an audit, not as a row-exclusion rule.

Actual missing income remains NA: 748 records in 2019 and 2,143 in 2025. Ordinary-income reference candidates (71,511 / 80,331) exclude endpoints and missing/invalid values but are **not** training IDs. The modeling team must first define the relevant training subset, then fit imputation/standardization/PCA within that subset (and each validation fold), applying frozen parameters to held-out and 2025 data. No imputer is fitted by this notebook.

The 103-column contract is 6 geography/provenance fields + 3 formula-label fields + 25×3 original/clean/QA columns + 13 derived-income fields + 6 official-label fields. It is not a 103-feature model matrix. Choose one target explicitly; do not use the other label or its provenance/comparison fields as predictors.


In [8]:
EXPECTED_COLUMNS = ['tract_id', 'State', 'County', 'meta_dataset_version', 'meta_population_census_year', 'meta_tract_vintage', 'fd_label', 'label_status', 'label_reason', 'original__Urban', 'clean__Urban', 'qa__Urban', 'original__population_total', 'clean__population_total', 'qa__population_total', 'original__occupied_housing_units_total', 'clean__occupied_housing_units_total', 'qa__occupied_housing_units_total', 'original__GroupQuartersFlag', 'clean__GroupQuartersFlag', 'qa__GroupQuartersFlag', 'original__NUMGQTRS', 'clean__NUMGQTRS', 'qa__NUMGQTRS', 'original__PCTGQTRS', 'clean__PCTGQTRS', 'qa__PCTGQTRS', 'original__TractKids', 'clean__TractKids', 'qa__TractKids', 'original__TractSeniors', 'clean__TractSeniors', 'qa__TractSeniors', 'original__TractWhite', 'clean__TractWhite', 'qa__TractWhite', 'original__TractBlack', 'clean__TractBlack', 'qa__TractBlack', 'original__TractAsian', 'clean__TractAsian', 'qa__TractAsian', 'original__TractNHOPI', 'clean__TractNHOPI', 'qa__TractNHOPI', 'original__TractAIAN', 'clean__TractAIAN', 'qa__TractAIAN', 'original__TractOMultir', 'clean__TractOMultir', 'qa__TractOMultir', 'original__TractHispanic', 'clean__TractHispanic', 'qa__TractHispanic', 'original__LowIncomeTracts', 'clean__LowIncomeTracts', 'qa__LowIncomeTracts', 'original__PovertyRate', 'clean__PovertyRate', 'qa__PovertyRate', 'original__MedianFamilyIncome', 'clean__MedianFamilyIncome', 'qa__MedianFamilyIncome', 'original__TractLOWI', 'clean__TractLOWI', 'qa__TractLOWI', 'original__TractHUNV', 'clean__TractHUNV', 'qa__TractHUNV', 'original__TractSNAP', 'clean__TractSNAP', 'qa__TractSNAP', 'original__LATracts_half', 'clean__LATracts_half', 'qa__LATracts_half', 'original__LATracts1', 'clean__LATracts1', 'qa__LATracts1', 'original__LATractsVehicle_20', 'clean__LATractsVehicle_20', 'qa__LATractsVehicle_20', 'original__low_access_population_1_20', 'clean__low_access_population_1_20', 'qa__low_access_population_1_20', 'income_value_status', 'income_source_missing', 'income_invalid_source', 'income_lower_endpoint', 'income_upper_endpoint', 'income_source_price_year', 'income_target_price_year', 'income_price_factor_applied', 'income_ordinary_2018_usd', 'income_with_endpoint_proxy_2018_usd', 'income_endpoint_proxy_applied', 'income_proxy_method', 'income_ordinary_reference_candidate']
FD2_COLUMNS = ['FD_2_original', 'FD_2', 'FD_2_source_field', 'FD_2_status', 'FD_2_reason', 'FD_2_comparison']
EXPECTED_COLUMNS += FD2_COLUMNS
for year, layer in cleaned.items():
    alternate = fd_alternative_tables[year]
    assert layer.tract_id.equals(alternate.tract_id)
    for column in FD2_COLUMNS:
        layer[column] = alternate[column]

final_rows, label_coverage = [], []
for year,layer in cleaned.items():
    assert layer.columns.tolist() == EXPECTED_COLUMNS
    assert len(layer)==len(base[year]) and layer.tract_id.is_unique
    assert layer.tract_id.str.fullmatch(r'[0-9]{11}').all()
    assert layer.label_status.isin(['known','unknown','not_applicable']).all()
    known = layer.label_status.eq('known')
    assert layer.loc[known,'fd_label'].isin(['0','1']).all()
    assert layer.loc[~known,'fd_label'].eq('').all()
    official_known = layer.FD_2_status.eq('known')
    assert layer.loc[official_known,'FD_2'].isin([0,1]).all()
    assert layer.loc[~official_known,'FD_2'].isna().all()
    comparable = known & official_known
    different = comparable & pd.to_numeric(layer.fd_label,errors='coerce').ne(layer.FD_2)
    assert layer.FD_2_comparison.eq('different').equals(different.fillna(False).astype(bool))
    for f in FIELDS:
        assert layer['original__'+f].equals(base[year][f])
    nonincome = ['clean__'+f for f in FIELDS if f not in ['MedianFamilyIncome','low_access_population_1_20']]
    for label_name, status_column in [('fd_label','label_status'),('FD_2','FD_2_status')]:
        for status in ['known','unknown','not_applicable']:
            mask = layer[status_column].eq(status)
            label_coverage.append({'year':year,'label':label_name,'status':status,'rows':int(mask.sum()),
                'income_missing':int((mask&layer.income_with_endpoint_proxy_2018_usd.isna()).sum()),
                'any_nonincome_candidate_missing':int((mask&layer[nonincome].isna().any(axis=1)).sum())})
    final_rows.append({'year':year,'rows':len(layer),'columns':len(layer.columns),'deleted_rows':0,
                       'statistically_imputed_values':0,'income_NA':int(layer.income_with_endpoint_proxy_2018_usd.isna().sum())})
assert all(file_hash(RAW_DATA_DIR/p)==h for p,h in input_hashes.items())
display(pd.DataFrame(final_rows))
display(pd.DataFrame(label_coverage))


,year,rows,columns,deleted_rows,statistically_imputed_values,income_NA
0,2019,72531,103,0,0,748
1,2025,84119,103,0,0,2143


,year,label,status,rows,income_missing,any_nonincome_candidate_missing
0,2019,fd_label,known,36617,377,10
1,2019,fd_label,unknown,35914,371,15
2,2019,fd_label,not_applicable,0,0,0
3,2019,FD_2,known,72531,748,25
4,2019,FD_2,unknown,0,0,0
5,2019,FD_2,not_applicable,0,0,0
6,2025,fd_label,known,83848,1873,433
7,2025,fd_label,unknown,0,0,0
8,2025,fd_label,not_applicable,271,270,271
9,2025,FD_2,known,83848,1873,433


## 8. Export Only the Two Cleaned CSVs

The output folder must not already exist. CSVs use UTF-8 with BOM and empty cells for numeric NA. Read them with `dtype=str, keep_default_na=False` first, then explicitly convert selected numeric columns; this preserves tract leading zeros and original source tokens.

The export is read back and checked against the in-memory table, column by column. A generated-file SHA-256 is displayed for reference, without writing another manifest or report file. Re-running this cell alone intentionally fails if its output folder exists; rerun setup for a new folder or explicitly choose a new path.

**Version change:** Six official-label fields are appended to the original 97-column schema. The original columns must remain unchanged; the new columns are verified against the source official flags. Both label-specific input-coverage summaries are recalculated. Routine execution does not require historical delivery files.


In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
export_rows = []
for year,layer in cleaned.items():
    path = OUTPUT_DIR/f'cleaned_{year}.csv'
    layer.to_csv(path,index=False,encoding='utf-8-sig',na_rep='')
    restored = pd.read_csv(path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    assert restored.columns.tolist()==EXPECTED_COLUMNS and len(restored)==len(layer)
    for column in EXPECTED_COLUMNS:
        if pd.api.types.is_numeric_dtype(layer[column].dtype):
            values = pd.to_numeric(restored[column].replace('',pd.NA),errors='raise').astype(layer[column].dtype)
            pd.testing.assert_series_equal(values,layer[column],check_names=False)
        else:
            pd.testing.assert_series_equal(restored[column],layer[column].fillna('').astype(str),check_dtype=False,check_names=False)
    export_rows.append({'year':year,'file':str(path),'rows':len(restored),'columns':len(restored.columns),'sha256':file_hash(path)})
assert {p.name for p in OUTPUT_DIR.iterdir()} == {'cleaned_2019.csv','cleaned_2025.csv'}
display(pd.DataFrame(export_rows))


,year,file,rows,columns,sha256
0,2019,/private/var/folders/4r/zlfp_lm51j11_zp5q_dl55...,72531,103,fe7b9e90f9eda9266062a2826b7c11cb1e23c2b56c1230...
1,2025,/private/var/folders/4r/zlfp_lm51j11_zp5q_dl55...,84119,103,c2ee925ef26fc82641d69e3b556b50e01267509d2efe69...
